In [1]:
#step1 : Importing libraries
import pandas as pd
import numpy as np

In [5]:
#step2 : loading dataset
df=pd.read_csv("C:/Users/sai meghana/Downloads/Task1/hotel_bookings.csv"

In [10]:
#step3 : Intial Analysing
print("Shape:", df.shape)

Shape: (119390, 33)


In [8]:
print("Missing values:\n", df.isnull().sum())

Missing values:
 hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340

In [18]:
df=df.drop('Unnamed: 32',axis=1)

In [19]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 19


In [20]:
print("Data types:\n",df.dtypes)

Data types:
 hotel                              object
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type                       object
agent                

In [21]:
#step4 : Handling duplicates
df=df.drop_duplicates()

In [22]:
#step5 : Handling missing values
#Filling numerical cols with median
num_cols=df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col]=df[col].fillna(df[col].median())
#Filling categorical cols with mode
cat_cols=df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col]=df[col].fillna(df[col].mode()[0])

In [23]:
#step6 : Standardizing date formats
df['reservation_status_date']=pd.to_datetime(df['reservation_status_date'],errors='coerce')

In [24]:
#step7 : Normalize categorical values
df['hotel']=df['hotel'].str.strip().str.title()
df['meal']=df['meal'].replace({'BB':'Bed & Breakfast','HB':'Half Board','SC':'Self Catering'})

In [26]:
#step8 : Feature Engineering
df['stay_length']=df['stays_in_weekend_nights']+df['stays_in_week_nights']
df['total_guests']=df['adults']+df['children'].fillna(0)+df['babies']
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'].astype(str) + '-' +
    df['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)
df.drop(['arrival_date_year','arrival_date_month',
         'arrival_date_week_number','arrival_date_day_of_month'], axis=1, inplace=True)

In [28]:
#step9 : Outlier Treatment
for col in ['adr','total_guests']:
    q1,q3=df[col].quantile([0.25,0.75])
    iqr=q3-q1
    lower,upper=q1-1.5*iqr,q3+1.5*iqr
    df[col]=np.where(df[col]<lower,lower,df[col])
    df[col]=np.where(df[col]>upper,upper,df[col])

In [29]:
#step10 : saving cleaned dataset
df.to_csv("cleaned_hotel_bookings.csv",index=False)

In [32]:
descriptions = {
    "hotel": "Type of hotel (City or Resort)",
    "is_canceled": "Whether booking was canceled (0 = No, 1 = Yes)",
    "lead_time": "Number of days between booking and arrival",
    "arrival_date_year": "Year of arrival",
    "arrival_date_month": "Month of arrival",
    "arrival_date_week_number": "Week number of arrival",
    "arrival_date_day_of_month": "Day of arrival",
    "stays_in_weekend_nights": "Nights stayed on weekends",
    "stays_in_week_nights": "Nights stayed on weekdays",
    "adults": "Number of adults",
    "children": "Number of children",
    "babies": "Number of babies",
    "meal": "Type of meal plan",
    "country": "Country of origin",
    "market_segment": "Market segment category",
    "distribution_channel": "Booking distribution channel",
    "is_repeated_guest": "Whether guest is repeated",
    "previous_cancellations": "Number of previous cancellations",
    "previous_bookings_not_canceled": "Number of previous non-canceled bookings",
    "reserved_room_type": "Reserved room type code",
    "assigned_room_type": "Assigned room type code",
    "booking_changes": "Number of changes made to booking",
    "deposit_type": "Type of deposit made",
    "agent": "Agent ID (if applicable)",
    "company": "Company ID (if applicable)",
    "days_in_waiting_list": "Days booking was on waiting list",
    "customer_type": "Type of customer",
    "adr": "Average Daily Rate",
    "required_car_parking_spaces": "Number of car parking spaces required",
    "total_of_special_requests": "Number of special requests",
    "reservation_status": "Final status of reservation",
    "reservation_status_date": "Date of reservation status",
    "Unnamed: 32": "Empty column (to be dropped)"
}

# Create dictionary DataFrame
data_dict = pd.DataFrame({
    "Column": df.columns,
    "DataType": [str(df[col].dtype) for col in df.columns],
    "Description": [descriptions.get(col, "No description available") for col in df.columns],
    "Example": [df[col].dropna().iloc[0] if df[col].notnull().any() else "NaN" for col in df.columns]
})

# Save to CSV
data_dict.to_csv("data_dictionary_with_examples.csv", index=False)